## Before you run
Select the `.venv` kernel created in the project folder. In the terminal, install dependencies using `.venv\Scripts\python.exe -m pip install -r requirements.txt`. Transformers must be version 4.57.1 for the classic summarization and translation task names used below.


# Week 7 – Hugging Face Pipelines

**Task:** Performing NLP Tasks using Hugging Face Pipelines

## Objective
- Use pretrained Transformer pipelines for common NLP tasks.
- Compare summarization, translation, sentiment analysis, and question answering.
- Save generated outputs in CSV files.

**Instructions:** Run the cells from top to bottom. The first run downloads pretrained models and needs an internet connection.

In [1]:
from pathlib import Path
import pandas as pd
from transformers import pipeline

ROOT = Path.cwd()
if not (ROOT / "dataset").exists() and (ROOT.parent / "dataset").exists():
    ROOT = ROOT.parent
DATA = ROOT / "dataset"
DATA.mkdir(exist_ok=True)
text = str(pd.read_csv(DATA / "input_text.csv").loc[0, "text"])
print("Article loaded. Characters:", len(text))


c:\Users\Lenovo T470s\Documents\Tezt Generator\.venv_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Article loaded. Characters: 3837


## 1. Sentiment Analysis

In [2]:
sentiment_pipe = pipeline("sentiment-analysis")
sentiment_result = sentiment_pipe(text[:2000])[0]
sentiment_df = pd.DataFrame([{
    "id": 1, "text": text[:2000], "label": sentiment_result["label"],
    "score": round(float(sentiment_result["score"]), 4), "status": "generated"
}])
sentiment_df.to_csv(DATA / "sentiment_results.csv", index=False)
sentiment_df[["label", "score"]]


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 999.42it/s] 


,label,score
0,NEGATIVE,0.9288


## 2. Summarization and comparison of max_length values

In [3]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
summary_rows = []
for max_len, min_len in [(70, 20), (120, 35)]:
    result = summarizer(text[:4000], max_length=max_len, min_length=min_len, do_sample=False)[0]
    summary_rows.append({
        "id": 1, "original_text": text, "summary": result["summary_text"],
        "max_length": max_len, "status": "generated"
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(DATA / "summarized_text.csv", index=False)
summary_df[["max_length", "summary"]]


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## 3. Translation (English to French)

In [ ]:
translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
translation = translator(text[:800])[0]["translation_text"]
translation_df = pd.DataFrame([{
    "id": 1, "source_language": "English", "target_language": "French",
    "original_text": text[:800], "translation": translation, "status": "generated"
}])
translation_df.to_csv(DATA / "translated_text.csv", index=False)
translation_df[["target_language", "translation"]]


## 4. Question Answering — five questions

In [ ]:
qa_pipe = pipeline("question-answering", model="distilbert/distilbert-base-cased-distilled-squad")
questions = [
    "What does sentiment analysis classify?",
    "Why should people verify AI-generated information?",
    "What does summarization help readers do?",
    "Which library provides the pipeline interface?",
    "Why can translation require expert checking?"
]
qa_rows = []
for question in questions:
    result = qa_pipe(question=question, context=text)
    qa_rows.append({
        "question": question, "context": text, "answer": result["answer"],
        "score": round(float(result["score"]), 4), "status": "generated"
    })
qa_df = pd.DataFrame(qa_rows)
qa_df.to_csv(DATA / "qa_results.csv", index=False)
qa_df[["question", "answer", "score"]]


## 5. Combined Pipeline Comparison

In [ ]:
comparison_rows = []
for _, row in summary_df.iterrows():
    comparison_rows.append({
        "task": "Summarization", "input": text[:250], "output": row["summary"],
        "observation": f"max_length={row['max_length']}", "status": "generated"
    })
comparison_rows.extend([
    {"task": "Translation", "input": text[:250], "output": translation,
     "observation": "English translated into French", "status": "generated"},
    {"task": "Sentiment analysis", "input": text[:250], "output": sentiment_result["label"],
     "observation": f"confidence={sentiment_result['score']:.4f}", "status": "generated"}
])
for row in qa_rows:
    comparison_rows.append({
        "task": "Question answering", "input": row["question"], "output": row["answer"],
        "observation": f"confidence={row['score']:.4f}", "status": "generated"
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(DATA / "pipeline_comparison.csv", index=False)
comparison_df


## 6. Conclusion

Hugging Face pipelines make it easier to use pretrained Transformer models for common NLP tasks. This project applies sentiment analysis, summarization, translation, and question answering to a sample article. Outputs are saved in CSV files and compared. Model results should always be checked because they may omit context or contain errors.

In [ ]:
print("Output files:")
for path in sorted(DATA.glob("*.csv")):
    print("-", path.name)
